In [14]:
import requests
import pandas as pd
import random

user_agent_list = [
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_5) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/13.1.1 Safari/605.1.15',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:77.0) Gecko/20100101 Firefox/77.0',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_5) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/83.0.4103.97 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:77.0) Gecko/20100101 Firefox/77.0',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/83.0.4103.97 Safari/537.36',
]

BASE_HEADERS = {
    "accept": "application/json, text/plain, */*",
    "accept-language": "it-IT,it;q=0.9,en-US;q=0.8,en;q=0.7",
    "origin": "https://www.legabasket.it",
    "priority": "u=1, i",
    "referer": "https://www.legabasket.it/",
    "sec-ch-ua": '"Chromium";v="130", "Google Chrome";v="130", "Not?A_Brand";v="99"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"macOS"',
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "cross-site",
}

def build_session(authorize=False):
    session = requests.Session()
    session.headers.update(BASE_HEADERS)
    if authorize:
        token = os.environ.get("LEGABASKET_TOKEN", "")
        session.headers["authorization"] = f"Bearer {token}"
    return session

def send_get_request(session, url):
    session.headers["user-agent"] = random.choice(user_agent_list)
    resp = session.get(url)
    resp.raise_for_status()
    return resp.json()

def _round2(val):
    return round(val, 2) if val is not None else None

def parse_player_row(row, team_name):

    return {
        'team':                team_name,
        'starter':             row.get('sf') == '1',
        'number':              row.get('player_num'),
        'player':              f"{row.get('player_surname', '')} {row.get('player_name', '')}".strip(),
        'points':              row.get('pun'),
        'minutes':             row.get('min'),
        'fouls_committed':     row.get('falli_c'),
        'fouls_received':      row.get('falli_sf'),
        'two_pts_made':        row.get('t2_r'),
        'two_pts_attempted':   row.get('t2_t'),
        'two_pts_perc':        _round2(row.get('t2_perc')),
        'dunks':               row.get('sc'),
        'three_pts_made':      row.get('t3_r'),
        'three_pts_attempted': row.get('t3_t'),
        'three_pts_perc':      _round2(row.get('t3_perc')),
        'ft_made':             row.get('tl_r'),
        'ft_attempted':        row.get('tl_t'),
        'ft_perc':             _round2(row.get('tl_perc')),
        'def_rebounds':        row.get('rimbalzi_d'),
        'off_rebounds':        row.get('rimbalzi_o'),
        'tot_rebounds':        row.get('rimbalzi_t'),
        'blocks_given':        row.get('stoppate_dat'),
        'blocks_received':     row.get('stoppate_sub'),
        'turnovers':           row.get('palle_p'),
        'steals':              row.get('palle_r'),
        'assists':             row.get('ass'),
        'league_rating':       row.get('val_lega'),
        'oer_rating':          _round2(row.get('val_oer')),
        'plus_minus':          row.get('plus_minus'),
    }

def parse_aggregate_row(row, team_name, label):
    
    return {
        'team':                team_name,
        'starter':             None,
        'number':              None,
        'player':              label,
        'points':              row.get('pun'),
        'minutes':             row.get('min'),
        'fouls_committed':     row.get('falli_c'),
        'fouls_received':      row.get('falli_sf'),
        'two_pts_made':        row.get('t2_r'),
        'two_pts_attempted':   row.get('t2_t'),
        'two_pts_perc':        _round2(row.get('t2_perc')),
        'dunks':               row.get('sc'),
        'three_pts_made':      row.get('t3_r'),
        'three_pts_attempted': row.get('t3_t'),
        'three_pts_perc':      _round2(row.get('t3_perc')),
        'ft_made':             row.get('tl_r'),
        'ft_attempted':        row.get('tl_t'),
        'ft_perc':             _round2(row.get('tl_perc')),
        'def_rebounds':        row.get('rimbalzi_d'),
        'off_rebounds':        row.get('rimbalzi_o'),
        'tot_rebounds':        row.get('rimbalzi_t'),
        'blocks_given':        row.get('stoppate_dat'),
        'blocks_received':     row.get('stoppate_sub'),
        'turnovers':           row.get('palle_p'),
        'steals':              row.get('palle_r'),
        'assists':             row.get('ass'),
        'league_rating':       row.get('val_lega'),
        'oer_rating':          _round2(row.get('val_oer')),
        'plus_minus':          row.get('plus_minus'),
    }

def scores_to_rows(resp):
    match = resp['match']
    scores = resp['scores']
    rows = []
    for side, team_key in [('ht', 'h_team_name'), ('vt', 'v_team_name')]:
        team_name = match[team_key]
        for player_row in scores[side].get('rows', []):
            rows.append(parse_player_row(player_row, team_name))
        rows.append(parse_aggregate_row(scores[side]['team'],   team_name, 'Squadra'))
        rows.append(parse_aggregate_row(scores[side]['totals'], team_name, 'Totali'))
    return rows

In [15]:
session = build_session()

In [19]:
all_game_ids = [25344]
print(f"Fetching details for {len(all_game_ids)} games...")

base_url = "https://www.legabasket.it/api/championships/get-championships-matches-by-id?id="
all_dfs = []

for game_id in all_game_ids:
    url = base_url + str(game_id)
    resp = send_get_request(session, url)

    player_rows = scores_to_rows(resp)
    df = pd.DataFrame(player_rows)
    df['legabasket_game_id'] = game_id
    all_dfs.append(df)

games_details_df = pd.concat(all_dfs, ignore_index=True)

Fetching details for 1 games...


In [20]:
games_details_df

,team,starter,number,player,points,minutes,fouls_committed,fouls_received,two_pts_made,two_pts_attempted,two_pts_perc,dunks,three_pts_made,three_pts_attempted,three_pts_perc,ft_made,ft_attempted,ft_perc,def_rebounds,off_rebounds,tot_rebounds,blocks_given,blocks_received,turnovers,steals,assists,league_rating,oer_rating,plus_minus,legabasket_game_id
0,Umana Reyer Venezia,True,1.0,COLE R.J.,27,31.0,1,6,6.0,7.0,85.71,0.0,2.0,5.0,40.00,9.0,10.0,90.00,3,0,3,0.0,0.0,3,1,6.0,34,1.35,3.0,25344
1,Umana Reyer Venezia,False,2.0,HORTON Chris,8,19.0,3,1,3.0,3.0,100.00,0.0,0.0,0.0,0.00,2.0,2.0,0.00,0,3,3,0.0,0.0,1,1,0.0,9,1.60,-2.0,25344
2,Umana Reyer Venezia,False,4.0,LEVER Alessandro,3,6.0,0,1,0.0,0.0,0.00,0.0,1.0,2.0,50.00,0.0,0.0,0.00,1,1,2,0.0,0.0,0,1,0.0,6,1.50,1.0,25344
3,Umana Reyer Venezia,False,5.0,DE NICOLAO Giovanni,0,0.0,0,0,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0,0,0,0.0,0.0,0,0,0.0,0,0.00,0.0,25344
4,Umana Reyer Venezia,False,7.0,CANDI Leonardo,3,11.0,2,1,0.0,0.0,0.00,0.0,1.0,2.0,50.00,0.0,0.0,0.00,1,0,1,0.0,0.0,1,0,0.0,1,1.00,-1.0,25344
5,Umana Reyer Venezia,True,8.0,BOWMAN Ky,10,29.0,2,2,3.0,4.0,75.00,1.0,1.0,3.0,33.33,1.0,2.0,50.00,3,0,3,1.0,0.0,3,0,4.0,11,0.91,9.0,25344
6,Umana Reyer Venezia,False,10.0,WHEATLE Carl,3,22.0,2,1,1.0,3.0,33.33,1.0,0.0,4.0,0.00,1.0,2.0,50.00,3,0,3,1.0,0.0,1,0,1.0,-1,0.33,0.0,25344
7,Umana Reyer Venezia,False,14.0,JANELIDZE Giga,0,0.0,0,0,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0,0,0,0.0,0.0,0,0,0.0,0,0.00,0.0,25344
8,Umana Reyer Venezia,True,22.0,PARKS Jordan,9,24.0,3,3,2.0,6.0,33.33,0.0,1.0,3.0,33.33,2.0,6.0,33.33,4,2,6,0.0,0.0,1,1,1.0,6,0.69,3.0,25344
9,Umana Reyer Venezia,True,33.0,WILTJER Kyle,3,21.0,1,4,1.0,1.0,100.00,0.0,0.0,1.0,0.00,1.0,3.0,33.33,2,0,2,0.0,0.0,0,0,0.0,5,0.86,4.0,25344
